# E7 - ViT cross-architecture validation of the feature-norm threshold

**Purpose.** The paper establishes the feature-norm threshold fn* on two
architectures (MLP-5, ResNet-20). This notebook adds a **third architecture, a
small Vision Transformer (ViT)**, under the *same* two-phase CE->MSE protocol,
the *same* NC1/NC2/NC3 definitions, and the *same* NC1<0.01 (strict) / NC1<0.05
(relaxed) criteria, with 3 seeds. If the ViT collapses to its own tight,
pair-specific fn* on MNIST, the threshold rule generalises beyond MLP/ResNet.

**Features measured.** NC is computed on the CLS-token vector that feeds the
linear classifier, consistent with the paper's rule of using the classifier
input.

**Honest caveats (must be reported with any result).**
1. **LayerNorm constrains the ViT feature norm.** The classifier input sits after
   a final LayerNorm, so its norm is of order sqrt(dim) and is held by the
   LayerNorm, unlike the *unnormalised* MLP/ResNet penultimate features whose norm
   freely compresses to fn*. Therefore the ViT fn* magnitude is **not directly
   comparable** to the MLP/ResNet values; the cross-architecture claim rests on
   whether the ViT collapses to its **own** tight, pair-specific fn* (low CV
   across seeds), not on where that value falls. The pre-LayerNorm CLS norm is
   also logged (`feat_norm_preln`) for transparency.
2. **Optimiser/warmup.** AdamW with a 5-epoch linear warmup (transformers are
   unstable without warmup): a documented deviation from the MLP/ResNet
   cosine-only schedule. Like the main grid, this cell does not isolate
   architecture from optimiser.
3. **Possible non-collapse.** A small ViT may not reach NC1<0.01 within the
   Phase-2 budget; it is then reported at NC1<0.05, or raise `phase2`. If it does
   not collapse at all, report that honestly (as for the CIFAR-100 attempt in the
   paper). **Do not fabricate or hand-edit fn* values.**

> Run on a GPU runtime. Kaggle: Settings -> Accelerator -> GPU T4, and Internet
> ON. MNIST is the primary cell; set `RUN_CIFAR10 = True` to add CIFAR-10.
> For crash safety use Kaggle "Save & Run All (Commit)"; per-seed CSVs and the
> summary are written to SAVE_DIR after each seed.

In [1]:
import torch, torchvision, time, os, sys
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.isdir('/kaggle/working'):
    PLATFORM, SAVE_DIR, DATA_DIR = 'kaggle', '/kaggle/working/', '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/nc_outputs/'
        print('Drive mounted ->', SAVE_DIR)
    except Exception as exc:
        print('WARNING: Drive mount failed (%s). Using /content/ (not crash-safe).' % exc)
        SAVE_DIR = '/content/'
    DATA_DIR = '/content/data/'
else:
    PLATFORM, SAVE_DIR, DATA_DIR = 'local', './', './data/'
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), 'No GPU - enable a GPU runtime (Kaggle: Settings > Accelerator > GPU T4).'
print('Platform:', PLATFORM, '| SAVE_DIR:', SAVE_DIR)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

Platform: kaggle | SAVE_DIR: /kaggle/working/
GPU: Tesla T4 | Torch: 2.10.0+cu128


In [2]:
# MNIST view for ViT: 1x28x28, patch 4 -> 49 tokens.
# CIFAR-10 view: 3x32x32, patch 4 -> 64 tokens. No augmentation (standard for NC).
# (Kaggle: enable Settings > Internet ON so torchvision can download the data.)
mnist_tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
cifar_tf = T.Compose([T.ToTensor(),
                      T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

def make_loaders(dataset, transform, batch_train=512, batch_test=1024):
    D = torchvision.datasets.MNIST if dataset == 'mnist' else torchvision.datasets.CIFAR10
    tr = D(DATA_DIR, train=True,  download=True, transform=transform)
    te = D(DATA_DIR, train=False, download=True, transform=transform)
    return (DataLoader(tr, batch_train, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(te, batch_test,  shuffle=False, num_workers=2, pin_memory=True))
print('loaders ready')

loaders ready


In [3]:
# ---- Small Vision Transformer (self-contained, no external deps) ----
# NC "features" = the CLS-token vector that feeds the linear classifier, i.e. the
# output AFTER the final LayerNorm (consistent with the paper's rule of using the
# classifier input). We ALSO record the pre-LayerNorm CLS norm as a diagnostic,
# because the final LayerNorm constrains the post-LN norm to ~sqrt(dim); see the
# caveat in the intro and the comparison cell.
class PatchEmbed(nn.Module):
    def __init__(self, in_ch, dim, patch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)   # B, N, dim

class ViT(nn.Module):
    def __init__(self, img_size, patch, in_ch, num_classes=10,
                 dim=128, depth=6, heads=4, mlp_ratio=2.0):
        super().__init__()
        self.patch = PatchEmbed(in_ch, dim, patch)
        n_tok = (img_size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, n_tok + 1, dim))
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads,
                    dim_feedforward=int(dim * mlp_ratio), dropout=0.0,
                    activation='gelu', batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        self._feats = None       # post-LN CLS (classifier input; used for NC)
        self._feats_pre = None   # pre-LN CLS (diagnostic only)
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)
    def forward(self, x):
        B = x.size(0)
        x = self.patch(x)
        x = torch.cat([self.cls.expand(B, -1, -1), x], dim=1) + self.pos
        x = self.blocks(x)
        self._feats_pre = x[:, 0].detach()        # CLS before final LayerNorm
        x = self.norm(x)
        post = x[:, 0]                            # CLS after final LayerNorm
        self._feats = post.detach()
        return self.head(post)
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

# ---- NC metrics + accuracy (NC1/NC2/NC3 identical to the main paper) ----
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval(); fl, pl, ll = [], [], []
    for x, y in loader:
        f = model.get_features(x.to(DEVICE, non_blocking=True))   # post-LN, classifier input
        fl.append(f.cpu()); ll.append(y)
        pre = getattr(model, '_feats_pre', None)
        if pre is not None: pl.append(pre.cpu())
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0); mu_c = torch.stack([H[Y == c].mean(0) for c in range(K)])
    M = mu_c - mu_G
    Sw = sum((H[Y == c] - mu_c[c]).T @ (H[Y == c] - mu_c[c]) for c in range(K)) / len(H)
    Sb = M.T @ M / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn = F.normalize(M, dim=1); cos = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2 = (cos[mask] - (-1. / (K - 1))).abs().mean().item()
    Wn = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3 = (1 - (Mn * Wn).sum(1).mean()).item()
    out = {'nc1': nc1, 'nc2': nc2, 'nc3': nc3, 'feat_norm': H.norm(dim=1).mean().item()}
    if pl:
        P = torch.cat(pl).float()
        out['feat_norm_preln'] = P.norm(dim=1).mean().item()
    return out

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / total
print('ViT + metrics ready')

ViT + metrics ready


In [4]:
import math
# Two-phase CE->MSE runner. Optimiser is AdamW (the natural choice for ViTs);
# a 5-epoch linear warmup is added because transformers are unstable without it.
# NOTE: this warmup is a documented deviation from the MLP/ResNet cosine-only
# schedule and must be reported as such. Like the main grid, the ViT cell does
# not isolate architecture from optimiser.
def make_opt_sched(params, n_ep, warm=5):
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
    def lr_lambda(ep):
        if ep < warm:
            return (ep + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (ep - warm) / max(1, n_ep - warm)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

def run_twophase(model, train_loader, test_loader, name,
                 phase1=200, phase2=600, nc_every=10, K=10):
    model = model.to(DEVICE); rows = []; terminal = False
    t_nc_strict = fn_strict = t_nc_relaxed = fn_relaxed = None
    t0 = time.time()
    for phase, loss_fn, n_ep in [(1, 'ce', phase1), (2, 'mse', phase2)]:
        opt, sch = make_opt_sched(model.parameters(), n_ep)
        off = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l; model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = (F.mse_loss(logits, F.one_hot(y, K).float())
                        if loss_fn == 'mse' else F.cross_entropy(logits, y))
                if not torch.isfinite(loss):
                    print('  [%s] non-finite loss at ep %d -> aborting' % (name, ep))
                    return pd.DataFrame(rows), t_nc_strict, fn_strict, t_nc_relaxed, fn_relaxed
                loss.backward(); opt.step()
            sch.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader); te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True; print('  [%s] terminal phase at ep %d (train acc %.4f)' % (name, ep, tr))
                nc = (compute_nc(model, train_loader, K) if (terminal or phase == 2)
                      else {'nc1': None, 'nc2': None, 'nc3': None, 'feat_norm': None})
                rows.append({'epoch': ep, 'phase': phase, 'train': tr, 'test': te, **nc})
                if nc['nc1'] is not None:
                    if t_nc_relaxed is None and nc['nc1'] < 0.05:
                        t_nc_relaxed, fn_relaxed = ep, nc['feat_norm']
                    if t_nc_strict is None and nc['nc1'] < 0.01:
                        t_nc_strict, fn_strict = ep, nc['feat_norm']
                        print('  [%s] NC1<0.01 at ep %d  fn=%.4f' % (name, ep, fn_strict))
    print('  [%s] done in %.1f min (final train %.4f, test %.4f)'
          % (name, (time.time() - t0) / 60,
             rows[-1]['train'] if rows else float('nan'),
             rows[-1]['test'] if rows else float('nan')))
    return pd.DataFrame(rows), t_nc_strict, fn_strict, t_nc_relaxed, fn_relaxed
print('runner ready')

runner ready


In [5]:
RUN_CIFAR10 = False   # set True to also run the ViT on CIFAR-10 (slower)
SEEDS = range(3)

def build(dataset):
    if dataset == 'mnist':
        tl, vl = make_loaders('mnist', mnist_tf)
        return ViT(img_size=28, patch=4, in_ch=1, dim=128, depth=6, heads=4), tl, vl
    tl, vl = make_loaders('cifar10', cifar_tf)
    return ViT(img_size=32, patch=4, in_ch=3, dim=192, depth=6, heads=6), tl, vl

CONFIG = ['mnist'] + (['cifar10'] if RUN_CIFAR10 else [])

rows = []
for dataset in CONFIG:
    for seed in SEEDS:
        tag = 'vit_%s_adam_s%d' % (dataset, seed)
        print('\n=== %s ===' % tag)
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
        model, tl, vl = build(dataset)
        print('  params: %.2fM' % (sum(p.numel() for p in model.parameters()) / 1e6))
        df, ts, fs, tr_, fr = run_twophase(model, tl, vl, tag)
        df.to_csv(SAVE_DIR + tag + '.csv', index=False)
        rows.append({'arch': 'vit', 'dataset': dataset, 'optimizer': 'adamw', 'seed': seed,
                     'T_NC_strict': ts, 'fn_strict': fs,
                     'T_NC_relaxed': tr_, 'fn_relaxed': fr,
                     'fn_preln_final': (df.feat_norm_preln.dropna().iloc[-1]
                                        if 'feat_norm_preln' in df and df.feat_norm_preln.notna().any()
                                        else None),
                     'test_acc_final': df.test.iloc[-1] if len(df) else None})
        pd.DataFrame(rows).to_csv(SAVE_DIR + 'vit_crossarch_summary.csv', index=False)

summ = pd.DataFrame(rows)
print('\n================ ViT CROSS-ARCH SUMMARY ================')
print(summ.to_string(index=False))


=== vit_mnist_adam_s0 ===


100%|██████████| 9.91M/9.91M [00:00<00:00, 31.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.26MB/s]
/tmp/ipykernel_23/2490884100.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


  params: 0.81M
  [vit_mnist_adam_s0] terminal phase at ep 20 (train acc 0.9931)
  [vit_mnist_adam_s0] done in 186.2 min (final train 1.0000, test 0.9885)

=== vit_mnist_adam_s1 ===
  params: 0.81M
  [vit_mnist_adam_s1] terminal phase at ep 30 (train acc 0.9941)
  [vit_mnist_adam_s1] done in 185.2 min (final train 1.0000, test 0.9893)

=== vit_mnist_adam_s2 ===
  params: 0.81M
  [vit_mnist_adam_s2] terminal phase at ep 20 (train acc 0.9902)
  [vit_mnist_adam_s2] done in 184.4 min (final train 1.0000, test 0.9876)

================ ViT CROSS-ARCH SUMMARY ================
arch dataset optimizer  seed T_NC_strict fn_strict  T_NC_relaxed  fn_relaxed  fn_preln_final  test_acc_final
 vit   mnist     adamw     0        None      None           410   12.527828      976.953430          0.9885
 vit   mnist     adamw     1        None      None           420   12.714295      744.836548          0.9893
 vit   mnist     adamw     2        None      None           400   12.774726      999.366211    

In [6]:
# Compare ViT fn* on MNIST against the paper's published values (same dataset).
PUBLISHED = {'mlp5_mnist': 1.052, 'resnet20_mnist': 5.867}

def cell_fn(dataset):
    s = summ[summ.dataset == dataset].dropna(subset=['fn_strict'])
    col, crit = 'fn_strict', 'NC1<0.01'
    if len(s) == 0:
        s = summ[summ.dataset == dataset].dropna(subset=['fn_relaxed'])
        col, crit = 'fn_relaxed', 'NC1<0.05'
    if len(s) == 0:
        return None
    v = s[col]
    cv = (v.std(ddof=1) / v.mean() * 100) if len(v) > 1 and v.mean() else float('nan')
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0), len(v), crit, cv

r = cell_fn('mnist')
if r is None:
    print('No ViT/MNIST collapse recorded at NC1<0.05 within the Phase-2 budget.')
    print('Report this honestly (as for the CIFAR-100 attempt), or raise phase2 and rerun.')
else:
    m, sd, n, crit, cv = r
    print('ViT-small / MNIST    fn* = %.3f +/- %.3f  (N=%d, %s, CV=%.1f%%)' % (m, sd, n, crit, cv))
    print('MLP-5     / MNIST    fn* = %.3f  (published)' % PUBLISHED['mlp5_mnist'])
    print('ResNet-20 / MNIST    fn* = %.3f  (published)' % PUBLISHED['resnet20_mnist'])
    print()
    print('READ THIS BEFORE REPORTING:')
    print('* The ViT fn* is measured at the classifier input, which sits AFTER a final')
    print('  LayerNorm, so its scale is ~sqrt(dim) and is constrained by LayerNorm, unlike')
    print('  the unnormalised MLP/ResNet features. Do NOT compare the ViT magnitude')
    print('  directly to MLP/ResNet; the cross-architecture claim rests on whether the ViT')
    print('  collapses to its OWN tight, pair-specific fn* (low CV across seeds).')
    print('* fn_preln_final (in the summary) is the pre-LayerNorm CLS norm, reported for')
    print('  transparency only.')

ViT-small / MNIST    fn* = 12.672 +/- 0.129  (N=3, NC1<0.05, CV=1.0%)
MLP-5     / MNIST    fn* = 1.052  (published)
ResNet-20 / MNIST    fn* = 5.867  (published)

READ THIS BEFORE REPORTING:
* The ViT fn* is measured at the classifier input, which sits AFTER a final
  LayerNorm, so its scale is ~sqrt(dim) and is constrained by LayerNorm, unlike
  the unnormalised MLP/ResNet features. Do NOT compare the ViT magnitude
  directly to MLP/ResNet; the cross-architecture claim rests on whether the ViT
  collapses to its OWN tight, pair-specific fn* (low CV across seeds).
* fn_preln_final (in the summary) is the pre-LayerNorm CLS norm, reported for
  transparency only.


In [7]:
import glob
out = sorted(glob.glob(SAVE_DIR + 'vit_*.csv'))
print('Outputs (%d):' % len(out)); [print(' ', f) for f in out]

Outputs (4):
  /kaggle/working/vit_crossarch_summary.csv
  /kaggle/working/vit_mnist_adam_s0.csv
  /kaggle/working/vit_mnist_adam_s1.csv
  /kaggle/working/vit_mnist_adam_s2.csv


[None, None, None, None]